In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Library installation
!pip install segmentation-models-pytorch

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torchvision import transforms
from PIL import Image
from skimage import morphology
from scipy.spatial import distance
from tqdm import tqdm
import matplotlib.pyplot as plt

# =============================================================================
# 1. CONFIGURATION PARAMETERS
# =============================================================================

# --- CONFIGURE PATHS HERE ---
CHECKPOINT_PATH = "/content/drive/MyDrive/SUBMISSION/BEST_MODEL_UnetPlusPlus.pt" # Path to .pt model checkpoint
test_images_dir = "/content/drive/MyDrive/Medical Image Processing codiviso/challenge retina/Dataset_vessel_stu/test/image"    # Path to test images directory
output_masks_dir = "/content/drive/MyDrive/SUBMISSION/maschere_predette"    # Path to output directory
# -------------------------------

# --- Model Parameters ---
MODEL_ARCH = "UnetPlusPlus"
BACKBONE = "resnet34"
INPUT_SIZE = (768, 768)       # Network input size
ORIGINAL_SIZE = (1024, 1024)  # Desired output size

# --- NORMALIZATION STATISTICS (Calculated on the training set) ---
CUSTOM_MEAN = [0.2894105613231659, 0.23288576304912567, 0.19715753197669983]
CUSTOM_STD = [0.19505903124809265, 0.14583425223827362, 0.1446814090013504]

# --- Preprocessing Parameters ---
PREPROC_CONFIG = {
    "APPLY_AVERAGING": True,
    "APPLY_CLAHE": True,
    "APPLY_GAMMA": True,

    # Specific values
    "KERNEL_SIZE": 3,
    "GAMMA_VALUE": 1.3,
    "CLAHE_CLIP": 2.0,
    "CLAHE_GRID": (8, 8)
}

# --- Post-processing Parameters ---
POST_CONFIG = {
    "THRESHOLD": 0.7,
    "MAX_DIST": 15,
    "MIN_AREA": 150,
    "DILATION_K": 3
}

# =============================================================================
# 2. PREPROCESSING FUNCTIONS
# =============================================================================

def apply_averaging(channel, kernel_size=3):
    """Applies a smoothing filter (Average Blur)."""
    return cv2.blur(channel, (kernel_size, kernel_size))

def apply_clahe(channel):
    """Applies CLAHE. Handles float->uint8->float conversion."""
    ch_float = channel.astype(np.float32)
    ch_min = ch_float.min()
    ch_max = ch_float.max()
    ch_norm = (ch_float - ch_min) / (ch_max - ch_min + 1e-8)
    ch_uint8 = (ch_norm * 255).astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=PREPROC_CONFIG["CLAHE_CLIP"],
                            tileGridSize=PREPROC_CONFIG["CLAHE_GRID"])
    img_clahe = clahe.apply(ch_uint8)

    return img_clahe.astype(np.float32) / 255.0

def gamma_corr(channel, gamma=1.3):
    """Applies Gamma Correction."""
    c = channel.astype(np.float32)
    if c.max() > 1.0:
        c = c / 255.0
    return c ** gamma

def preprocess_image_pipeline(pil_image):
    """
    Applies the full pipeline: Averaging -> CLAHE -> Gamma -> Network Normalization
    """
    # 1. PIL -> Numpy -> Float [0, 1] Conversion
    img_np = np.array(pil_image)
    img_float = img_np.astype(np.float32) / 255.0

    channels = cv2.split(img_float)
    processed_channels = []

    # 2. Modular Pipeline for each channel
    for ch in channels:
        temp_ch = ch.copy()

        # Step A: Averaging (Smoothing)
        if PREPROC_CONFIG["APPLY_AVERAGING"]:
            temp_ch = apply_averaging(temp_ch, PREPROC_CONFIG["KERNEL_SIZE"])

        # Step B: CLAHE
        if PREPROC_CONFIG["APPLY_CLAHE"]:
            temp_ch = apply_clahe(temp_ch)

        # Step C: Gamma Correction
        if PREPROC_CONFIG["APPLY_GAMMA"]:
            temp_ch = gamma_corr(temp_ch, PREPROC_CONFIG["GAMMA_VALUE"])

        processed_channels.append(temp_ch)

    # 3. RGB Reconstruction
    merged_rgb = cv2.merge(processed_channels)
    merged_rgb = np.clip(merged_rgb, 0, 1) # Safety clipping

    # 4. Preparation for PyTorch (Resize + Norm + ToTensor)
    processed_pil = Image.fromarray((merged_rgb * 255).astype(np.uint8))

    final_transform = transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=CUSTOM_MEAN, std=CUSTOM_STD)
    ])

    return final_transform(processed_pil)

# =============================================================================
# 3. POST-PROCESSING FUNCTIONS
# =============================================================================

def get_endpoints(skeleton):
    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    filtered = cv2.filter2D(skeleton.astype(np.uint8), -1, kernel)
    return np.argwhere(filtered == 11)

def apply_gap_repair(mask_bin, max_dist, min_area, dilation_k):
    # Skeletonize
    skel = morphology.skeletonize(mask_bin > 0).astype(np.uint8)
    endpoints = get_endpoints(skel)

    final_mask = mask_bin.copy()

    # Gap reconnection
    if len(endpoints) >= 2:
        dists = distance.cdist(endpoints, endpoints, metric='euclidean')
        np.fill_diagonal(dists, np.inf)

        connections = np.zeros_like(mask_bin)
        processed = set()

        for i in range(len(endpoints)):
            if i in processed: continue

            nearest_idx = np.argmin(dists[i])
            dist_val = dists[i, nearest_idx]

            if dist_val <= max_dist:
                p1, p2 = endpoints[i], endpoints[nearest_idx]
                cv2.line(connections, (p1[1], p1[0]), (p2[1], p2[0]), 1, thickness=1)

                processed.add(i); processed.add(nearest_idx)
                dists[i,:] = np.inf; dists[:,i] = np.inf
                dists[nearest_idx,:] = np.inf; dists[:,nearest_idx] = np.inf

        # Connection dilation
        if dilation_k > 0:
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_k, dilation_k))
            connections = cv2.dilate(connections, kernel)

        final_mask = cv2.bitwise_or(mask_bin, connections)

    # Removal of small objects
    if min_area > 0:
        final_mask = morphology.remove_small_objects(final_mask.astype(bool), min_size=min_area).astype(np.uint8)

    return final_mask

# =============================================================================
# 4. MAIN: MODEL LOADING AND MASKS GENERATION
# =============================================================================

os.makedirs(output_masks_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model Loading
print(f"Loading {MODEL_ARCH} (Backbone: {BACKBONE})...")
model = smp.UnetPlusPlus(
    encoder_name=BACKBONE,
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
)

try:
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    print("✅ Weights loaded.")
except FileNotFoundError:
    print(f"❌ Error: File {CHECKPOINT_PATH} not found.")

model.to(device)
model.eval()

# Loop over images
image_files = sorted([f for f in os.listdir(test_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))])
print(f"Processing {len(image_files)} images...")

for img_name in tqdm(image_files):
    img_path = os.path.join(test_images_dir, img_name)

    # 1. Loading and Preprocessing
    image_pil = Image.open(img_path).convert('RGB')
    input_tensor = preprocess_image_pipeline(image_pil)
    input_tensor = input_tensor.unsqueeze(0).to(device)

    # 2. Inference
    with torch.no_grad():
        output = model(input_tensor)
        # Resize output to 1024x1024 BEFORE post-processing
        output = torch.nn.functional.interpolate(output, size=ORIGINAL_SIZE, mode='bilinear', align_corners=False)
        prob_mask = torch.sigmoid(output).cpu().squeeze().numpy()

    # 3. Binarization
    mask_bin = (prob_mask > POST_CONFIG["THRESHOLD"]).astype(np.uint8)

    # 4. Post-Processing (Repair & Clean)
    mask_final = apply_gap_repair(
        mask_bin,
        max_dist=POST_CONFIG["MAX_DIST"],
        min_area=POST_CONFIG["MIN_AREA"],
        dilation_k=POST_CONFIG["DILATION_K"]
    )

    # 5. Saving
    pred_mask_img = Image.fromarray(mask_final * 255)
    pred_mask_img.save(os.path.join(output_masks_dir, img_name))

print(f"\n✅ Finished! Masks saved in: {output_masks_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading UnetPlusPlus (Backbone: resnet34)...
✅ Weights loaded.
Processing 40 images...


100%|██████████| 40/40 [05:32<00:00,  8.32s/it]


✅ Finished! Masks saved in: /content/drive/MyDrive/SUBMISSION/maschere_predette
